# LightGBM 实验

[English](../../05_lightgbm.ipynb) · 简体中文

E3 预测 log2 半衰期，E4 直接预测回忆率。


In [1]:
# Locate shared data and modules when running from a subdirectory.
from pathlib import Path
import os
import sys

_notebook_start = Path.cwd().resolve()
_notebook_parents = (_notebook_start, *_notebook_start.parents)
_project_root = next(
    (p for p in _notebook_parents
     if (p / "data/raw/settles.acl16.learning_traces.13m.csv.gz").is_file()),
    None,
)
if _project_root is None:
    _project_root = next(
        (p for p in _notebook_parents if (p / "experiment_pipeline.py").is_file()),
        _notebook_start,
    )
os.chdir(_project_root)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from experiment_pipeline import Pipeline
run = Pipeline(language="zh-CN")
globals().update(run.lightgbm())


原始文件： D:\fujing123\Projects\Data-Analytic\HLR\data\raw\settles.acl16.learning_traces.13m.csv.gz
前 5 行：


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
0,1.0,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,2,2
1,0.5,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,2,1
2,1.0,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,1,1
3,0.5,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,2,1
4,1.0,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,1,1


Reused saved duplicate check and cleaned dataset.


,raw_rows,exact_duplicates_removed,clean_rows
0,12854226,81,12854145


Model and EDA input: D:\fujing123\Projects\Data-Analytic\HLR\data\processed\settles.acl16.learning_traces.13m.dedup_v1.csv.gz


,split,start,stop_exclusive,rows
0,Train,0,9254984,9254984
1,Validation,9254984,11568730,2313746
2,Test,11568730,12854145,1285415


切分依据：原始 CSV 文件行顺序；随机打乱仅发生在 Train 内的训练阶段。
已复用与当前原始数据、切分一致的特征缓存。
缓存：D:\fujing123\Projects\Data-Analytic\HLR\data\interim\takeaway_hlr_features_v1.dat；539.4 MiB
基础特征： ['sqrt(1+history_correct)', 'sqrt(1+history_seen-history_correct)', 'bias']
Train 词项数：18,353；Validation 未见词项行数：1,150；Test 未见词项行数：560
Reused Train-only categorical index cache.
Train categories: 92,392 users, 6 learning languages, 4 UI languages.
Validation unseen users: 590,955


Loaded lgbm results: D:\fujing123\Projects\Data-Analytic\HLR\data\artifacts\lgbm-9ddfecffb33c8fce.joblib


# 6. LightGBM 实验

## 6.1 E3：LGBM-logh 预测半衰期

E3 使用与 E1 相同的输入和 `log2(h_observed)` 标签。六组候选在相同的 20 万条 Train 上训练，使用完整 Validation 的目标均方误差提前停止，再按 Validation `MAE(p)` 选参。选定后在全部 Train 上重训；Test 留到第 8 节。

In [2]:
from IPython.display import display
print("lgbm_e3_candidate_results")
display(lgbm_e3_candidate_results)
print("lgbm_e3_validation")
display(lgbm_e3_validation)
print("lgbm_e3_resource")
display(lgbm_e3_resource)


lgbm_e3_candidate_results


,num_leaves,min_child_samples,best_iteration,n_train,n_validation,fit_seconds,predict_seconds,MAE(p)
0,15,50,94,200000,2313746,1.954245,0.431088,0.160506
1,15,200,91,200000,2313746,65.715613,0.441909,0.160547
2,31,50,65,200000,2313746,1.738091,0.361899,0.160559
3,31,200,73,200000,2313746,2.289201,0.390789,0.160601
4,63,50,53,200000,2313746,2.070233,0.338451,0.160641
5,63,200,61,200000,2313746,3.544870,0.379716,0.160676


lgbm_e3_validation


,model,n_train,n_validation,MAE(p),RMSE(p),AUC,Spearman(h),n_spearman_h,n_h_interior,h_interior_share,Spearman(h)_interior,p_hat_at_bounds,h_hat_at_bounds
0,LGBM-logh (E3),9254984,2313746,0.160429,0.314646,0.548446,0.16306,2313746,1025825,0.443361,0.045099,0.240425,0.0


lgbm_e3_resource


,n_train,num_leaves,min_child_samples,best_iteration,fit_seconds,predict_seconds,rss_before_GiB,peak_rss_GiB
0,9254984,15,50,271,159.005475,0.988067,0.287064,1.288982


## 6.2 E4：LGBM-p 直接预测回忆率

E4 使用与 E2 相同的四项输入 `[x_correct, x_wrong, bias, delta_days]` 和 `p_target` 标签。六组候选在相同的 20 万条 Train 上拟合，按完整 Validation 的目标均方误差提前停止，再按 Validation `MAE(p)` 选参。选定后在全部 Train 上重训；`h_hat` 由 `p_hat` 和 `delta_days` 反推。

In [3]:
from IPython.display import display
print("lgbm_e4_candidate_results")
display(lgbm_e4_candidate_results)
print("lgbm_e4_validation")
display(lgbm_e4_validation)
print("lgbm_e4_resource")
display(lgbm_e4_resource)


lgbm_e4_candidate_results


,num_leaves,min_child_samples,best_iteration,n_train,n_validation,fit_seconds,predict_seconds,MAE(p)
0,15,200,136,200000,2313746,1.530961,0.550972,0.170633
1,15,50,124,200000,2313746,2.237114,0.568140,0.170642
2,31,200,92,200000,2313746,1.279685,0.474607,0.170649
3,31,50,79,200000,2313746,0.983341,0.402051,0.170659
4,63,200,70,200000,2313746,1.452273,0.523379,0.170715
5,63,50,60,200000,2313746,1.077003,0.444318,0.170726


lgbm_e4_validation


,model,n_train,n_validation,MAE(p),RMSE(p),AUC,Spearman(h),n_spearman_h,n_h_interior,h_interior_share,Spearman(h)_interior,p_hat_at_bounds,h_hat_at_bounds
0,LGBM-p (E4),9254984,2313746,0.16997,0.267656,0.608162,0.624405,2313746,1025825,0.443361,0.023734,0.00001,0.068119


lgbm_e4_resource


,n_train,num_leaves,min_child_samples,best_iteration,fit_seconds,predict_seconds,rss_before_GiB,peak_rss_GiB
0,9254984,15,200,333,17.323614,1.010204,0.74152,1.420925
